In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
df = pd.read_csv("../data/german_credit_data.csv")

### Pré processamento

In [5]:
# Remover coluna de índice

df = df.drop(columns="Unnamed: 0")

print(df.columns)
print(f"\nNúmero de colunas: {df.shape[1]}")

Index(['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account',
       'Credit amount', 'Duration', 'Purpose', 'Risk'],
      dtype='object')

Número de colunas: 10


In [6]:
# Tratamento dos valores ausentes

df["Saving accounts"] = df["Saving accounts"].fillna("No Saving Account")

df["Checking account"] = df["Checking account"].fillna("No Checking Account")

print("Valores ausentes após o tratamento:\n")
print(df.isnull().sum())

print("\nCategorias de Saving accounts:")
print(df["Saving accounts"].value_counts())

print("\nCategorias de Checking account:")
print(df["Checking account"].value_counts())

Valores ausentes após o tratamento:

Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64

Categorias de Saving accounts:
Saving accounts
little               603
No Saving Account    183
moderate             103
quite rich            63
rich                  48
Name: count, dtype: int64

Categorias de Checking account:
Checking account
No Checking Account    394
little                 274
moderate               269
rich                    63
Name: count, dtype: int64


In [7]:
# Separação entre variáveis preditoras e alvo

X = df.drop(columns="Risk")

y = df["Risk"]

print(f"Dimensão de X: {X.shape}")
print(f"Dimensão de y: {y.shape}")

print("\nVariáveis preditoras:")
display(X.head())

print("\nVariável alvo:")
display(y.head())

Dimensão de X: (1000, 9)
Dimensão de y: (1000,)

Variáveis preditoras:


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,67,male,2,own,No Saving Account,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,No Checking Account,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car



Variável alvo:


0    good
1     bad
2    good
3    good
4     bad
Name: Risk, dtype: object

In [8]:
# Codificação da variável alvo

y = y.map({
    "good": 0,
    "bad": 1
})
# One-Hot Encoding

X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

# Verificações

print(f"Dimensão de X: {X.shape}")
print(f"Dimensão de y: {y.shape}")

print("\nPrimeiras linhas de X:")
display(X.head())

print("\nDistribuição da variável alvo:")
print(y.value_counts())

Dimensão de X: (1000, 21)
Dimensão de y: (1000,)

Primeiras linhas de X:


,Age,Job,Credit amount,Duration,Sex_male,Housing_own,Housing_rent,Saving accounts_little,Saving accounts_moderate,Saving accounts_quite rich,Saving accounts_rich,Checking account_little,Checking account_moderate,Checking account_rich,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
0,67,2,1169,6,1,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0
1,22,2,5951,48,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0
2,49,1,2096,12,1,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,45,2,7882,42,1,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0
4,53,2,4870,24,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0



Distribuição da variável alvo:
Risk
0    700
1    300
Name: count, dtype: int64


In [9]:
# Divisão entre treino e teste

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensões dos conjuntos:\n")

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

print(f"\ny_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

print("\nDistribuição da variável alvo (%)")

print("\nTreino")
print(y_train.value_counts(normalize=True).round(3) * 100)

print("\nTeste")
print(y_test.value_counts(normalize=True).round(3) * 100)

Dimensões dos conjuntos:

X_train: (800, 21)
X_test : (200, 21)

y_train: (800,)
y_test : (200,)

Distribuição da variável alvo (%)

Treino
Risk
0    70.0
1    30.0
Name: proportion, dtype: float64

Teste
Risk
0    70.0
1    30.0
Name: proportion, dtype: float64


## Criando Modelo

In [10]:
RandomForestClassifier(random_state=42)

RandomForestClassifier(random_state=42)

In [11]:
# Modelo Baseline

rf = RandomForestClassifier(
    random_state=42
)

# Treinamento
rf.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [12]:
# Classes previstas
y_pred = rf.predict(X_test)

# Probabilidade da classe positiva (Bad = 1)
y_prob = rf.predict_proba(X_test)[:, 1]

predicoes = pd.DataFrame({
    "Real": y_test.values,
    "Previsto": y_pred,
    "Probabilidade_Bad": y_prob
})

display(predicoes.head(10))

,Real,Previsto,Probabilidade_Bad
0,0,0,0.08
1,0,0,0.28
2,1,1,0.55
3,0,1,0.71
4,1,0,0.24
5,0,0,0.36
6,0,0,0.20
7,0,0,0.38
8,0,0,0.41
9,0,0,0.24


## Ajustes hiperparâmetros

In [13]:
# Modelo base


rf_tuning = RandomForestClassifier(
    random_state=42
)

# Espaço de hiperparâmetros


param_grid = {

    "n_estimators": [100, 200, 300, 500],

    "max_depth": [
        None,
        5,
        10,
        15,
        20
    ],

    "min_samples_split": [
        2,
        5,
        10
    ],

    "min_samples_leaf": [
        1,
        2,
        5,
        10
    ],

    "max_features": [
        "sqrt",
        "log2"
    ],

    "class_weight": [
        None,
        "balanced"
    ]

}

# Randomized Search

random_search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_grid,
    n_iter=50,
    scoring="recall",
    cv=5,
    random_state=42,
    n_jobs=-1
)


print("Configuração criada!")

Configuração criada!


In [14]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'class_weight': [None, 'balanced'],
                                        'max_depth': [None, 5, 10, 15, 20],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 5, 10],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='recall')

In [15]:
best_rf = random_search.best_estimator_

# Predições
y_pred_tuned = best_rf.predict(X_test)

y_prob_tuned = best_rf.predict_proba(X_test)[:,1]

print("Modelo otimizado pronto!")

Modelo otimizado pronto!


In [16]:
# Métricas modelo otimizado

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

precision_tuned = precision_score(y_test, y_pred_tuned)

recall_tuned = recall_score(y_test, y_pred_tuned)

f1_tuned = f1_score(y_test, y_pred_tuned)

roc_auc_tuned = roc_auc_score(y_test, y_prob_tuned)


print("Modelo Otimizado\n")

print(f"Accuracy : {accuracy_tuned:.3f}")
print(f"Precision: {precision_tuned:.3f}")
print(f"Recall   : {recall_tuned:.3f}")
print(f"F1-score : {f1_tuned:.3f}")
print(f"ROC-AUC  : {roc_auc_tuned:.3f}")


print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        y_pred_tuned,
        target_names=["Good", "Bad"]
    )
)

Modelo Otimizado

Accuracy : 0.730
Precision: 0.537
Recall   : 0.733
F1-score : 0.620
ROC-AUC  : 0.770

Classification Report

              precision    recall  f1-score   support

        Good       0.86      0.73      0.79       140
         Bad       0.54      0.73      0.62        60

    accuracy                           0.73       200
   macro avg       0.70      0.73      0.71       200
weighted avg       0.77      0.73      0.74       200



## Importancia das variáveis

In [95]:
# Usando o best_rf em vez do rf baseline
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

display(feature_importance.head(15))

,Feature,Importance
3,Duration,0.187937
2,Credit amount,0.163268
11,Checking account_little,0.157123
0,Age,0.109950
7,Saving accounts_little,0.078573
12,Checking account_moderate,0.075504
5,Housing_own,0.042217
18,Purpose_radio/TV,0.036585
1,Job,0.030523
4,Sex_male,0.029695


In [ ]:
# Gráfico de importância

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature",
    hue="Feature",
    legend=False,
    palette="viridis" 
)

plt.title("Top 15 Variáveis Mais Importantes - Random Forest Otimizada")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

### Salvando Modelo

In [97]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
os.makedirs("../processed", exist_ok=True)

joblib.dump(rf, "../models/random_forest_baseline.pkl")
joblib.dump(best_rf, "../models/random_forest_tuned.pkl")

joblib.dump(X_train, "../processed/X_train.pkl")
joblib.dump(X_test, "../processed/X_test.pkl")
joblib.dump(y_train, "../processed/y_train.pkl")
joblib.dump(y_test, "../processed/y_test.pkl")

print("Modelos e dados exportados com sucesso!")

Modelos e dados exportados com sucesso!


In [98]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

In [99]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    best_rf,
    "../models/random_forest_tuned.pkl"
)

['../models/random_forest_tuned.pkl']